# 1. Extended thinking

讓 Claude 在回覆前花時間對 task 進行思考，適合複雜的任務使用，需要多消耗時間以及 token

> [使用限制](https://docs.anthropic.com/en/docs/build-with-claude/extended-thinking#feature-compatibility)

## How Extended Thinking Works

開啟 extended thinking 功能後，Claude 回覆的內容會包含兩部分:
1. `Text Block` -> Final answer
2. `Thinking Block` -> Reasoning process

![](./figure/05/thinking_01.jpg)

有以下優點以及缺點:
- 優點:
    - Better reasoning capabilities for complex tasks
    - Increased accuracy on difficult problems
    - Transparency into Claude's thought process
- 缺點:
    - Higher costs (you pay for thinking tokens)
    - Increased latency (thinking takes time)
    - More complex response handling in your code

## When to Use Extended Thinking

當用盡一般的 prompt 技巧以及方法後，產生的內容若還是不如你的預期(或沒有達到你要的標準)，此時就可以使用

> 官方原話:
> Run your prompts without thinking first (先不用思考功能，不是叫你不要思考) , and if the accuracy isn't meeting your requirements after you've already optimized your prompt, then consider enabling extended thinking. It's a tool for when standard prompting isn't quite getting you there.

## Response Structure and Security

Claude 會在回覆 thinking 訊息內容中加入一個 signature，來確保後續你在回傳 thinking 紀錄給 Claude 時，內容沒有竄改。
這可以防止 developer 汙染 thinking 內容，避免 Claude 被操控或回覆不安全的內容

![](./figure/05/thinking_02.jpg)

## Redacted Thinking

有時候使用 thinking 時會收到 unreadable 的 Thinking Block，這表示 thinking 過程被 Claude 內部安全系統 flag，並將 原先的 thinking process plaintext 進行加密。後續要當成 thinking 紀錄使用時將這些加密後的內容傳回去即可。

![](./figure/05/thinking_03.png)

## Implementation

詳見 [./source/001_thinking_complete.ipynb](./source/001_thinking_complete.ipynb)

# 2. Image support

Claude 擁有 vision process 能力，可以使用他來處理圖片問題，像是:
- describe what's in an image, 
- compare multiple images, 
- count objects, 
- perform complex visual analysis tasks

## Image Handling Basics

處理圖片時有以下幾個限制需要特別注意:
1. Up to 100 images across all messages in a single request
1. Max size of 5MB per image
1. When sending one image: max height/width of 8000px
1. When sending `multiple` images: max height/width of 2000px
1. Images can be included as `base64` encoding or a `URL` to the image
1. Each image counts as tokens based on its dimensions: `tokens = (width px × height px) / 750`

![](./figure/05/image_01.jpg)

要把圖片傳給 Claude 處理時，需要加入 `Image Block`

架構會像這樣:
```python
with open("image.png", "rb") as f:
    image_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

add_user_message(messages, [
    # Image Block
    {
        "type": "image",
        "source": {
            "type": "base64",
            "media_type": "image/png",
            "data": image_bytes,
        }
    },
    # Text Block
    {
        "type": "text",
        "text": "What do you see in this image?"
    }
])
```

Claude 處理後會回傳 Text Block

![](./figure/05/image_02.jpg)

## Prompting Techniques

在處理圖片時，需要好的 prompt engineering 技術。過於簡單的文字描述往往會收到不好的結果

![](./figure/05/image_03.jpg)

可以透過以下方法來大幅增加 Claude 回覆準確度:
- Providing **detailed guidelines** and **analysis steps**
- Using one-shot or multi-shot **examples**
- Breaking down complex tasks into **smaller steps**

### 1. Step-by-Step Analysis

提供 claude 一步步分析的方法，例如:

```python
"""
Analyze this image of marbles and determine the exact count using this methodology:
1. Begin by identifying each unique marble one at a time. Assign each a number as you identify it.
2. Verify your result by counting with a different method. Start from the bottom-left corner and work row by row, from left to right.

What is the exact, verified number of marbles in this image?
"""
```

![](./figure/05/image_04.jpg)

### 2. One-Shot Examples

![](./figure/05/image_05.jpg)

## Real-World Example: Fire Risk Assessment

設計一個 *使用衛星圖片來分析家屋因森林大火而發生火災風險* 的系統，這個系統需要辨別以下事情:
- Dense, close-packed trees near the residence
- Difficult access routes for emergency services
- Branches overhanging the residence

![](./figure/05/image_06.jpg)

相較於簡單的 *提供火災風險分數* 這樣籠統的 prompt，撰寫擁有架構、將分析方法拆解成較小步驟的 prompt 更為合適，例如:
```python
"""
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75%+)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure

4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the home
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple vulnerability points
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numerical rating.
"""
```

剩下的程式碼可以看: [./source/002_images.ipynb](./source/002_images.ipynb)

# 3. PDF Support

簡單修改使用 Image 的程式碼即可
> 詳見 [./source/002_images.ipynb 最後一格](./source/002_images.ipynb)

## Key Changes from Image Processing

- Change the file extension from `.png` to `.pdf`
- Update the variable name from `image_bytes` to `file_bytes` for clarity
- Set the type to **`document`** instead of **`image`**
- Change the media type to **`application/pdf`** instead of **`image/png`**

```python
with open("earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []

add_user_message(
    messages,
    [
        {
            "type": "document",  # <- 修改
            "source": {
                "type": "base64",
                "media_type": "application/pdf", # <- 修改
                "data": file_bytes, # <- 因應變數名稱變更而修改
            },
        },
        {"type": "text", "text": "Summarize the document in one sentence"}, # <- 修改 prompt 內容
    ],
)

chat(messages)
```

## What Claude Can Extract from PDFs

- **Text content** throughout the document
- **Images and charts** embedded in the PDF
- **Tables** and their **data relationships**
- Document structure and formatting

> This makes Claude essentially a one-stop solution for extracting any type of information from PDF documents, whether you need summaries, data analysis, or specific content extraction.

# 4. Citation

> 詳見 [./source/002_citations_complete.ipynb](./source/002_citations_complete.ipynb)

## Enabling Citations

開啟 Citation 功能需要更改 message structure 內容:
- `title` 欄位需要填入一個 Claude 可閱讀的名稱 
    > The title field gives your document a readable name
- 新增 `citations: {"enabled": True}` 欄位

```python
{
    "type": "document",
    "source": {
        "type": "base64",
        "media_type": "application/pdf",
        "data": file_bytes,
    },
    "title": "earth.pdf",
    "citations": { "enabled": True }
}
```

## Understanding Citation Structure

Each citation contains several key pieces of information:

- `cited_text` - The exact text from your document that supports Claude's statement
- `document_index` - Which document Claude is referencing (useful when you provide multiple documents)
- `document_title` - The title you assigned to the document
- `start_page_number` - Where the cited text begins
- `end_page_number` - Where the cited text ends

![](./figure/05/citation_01.jpg)

後續可以再加工產生使用者可以互動的網頁，較為直覺也較好查驗

![](./figure/05/citation_02.jpg)

## Citations with Plain Text

Citation 不限於 pdf 檔案使用，也可用在 plaintext 上，需要將 message structure 修改成以下格式:

```python
{
    "type": "document", 
    "source": {
        "type": "text", # <- 修改
        "media_type": "text/plain", # <- 修改
        "data": article_text, # <- 根據變數名稱修改
    },
    "title": "earth_article", # <- 修改
    "citations": { "enabled": True }
}
```

# 5. Prompt Caching

Claude 在處理完每一筆 request 後都會將處理過程產生的資訊丟掉，但當使用者帶著上次剛處理完的資訊進一步 request Claude時， Claude 又得再處理一次，多花了時間

Prompt Caching 技術可以將 Claude 處理完的資訊進行暫存，短時間內如果相同資訊再來一次就可以不用重新處理
> The cache lives for **one hour**, so this feature is only useful if you're repeatedly sending the same content within that timeframe.

## Normally Processes Requests

將 request 傳給 Claude 後，Claude 並不會馬上開始處理，而是會做以下手續:
- Tokenizes the prompt into smaller pieces
- Creates embeddings for each token
- Adds context based on surrounding text
- Only then generates the actual output text

![](./figure/05/Prompt_Caching_01.jpg)
![](./figure/05/Prompt_Caching_02.jpg)

將訊息回覆給 user 之後，Claude 會將計算工作摻生的內容都丟棄

![](./figure/05/Prompt_Caching_03.jpg)

## The Problem with Discarding Work

如果這時 user 發送了新的 follow-up request ， Claude 又得重新處理一次

![](./figure/05/Prompt_Caching_04.jpg)
![](./figure/05/Prompt_Caching_05.jpg)

## How Prompt Caching Solves This

Prompt caching changes this workflow by saving the preprocessing work instead of discarding it

![](./figure/05/Prompt_Caching_06.jpg)
![](./figure/05/Prompt_Caching_07.jpg)

## Key Benefits and Limitations

![](./figure/05/Prompt_Caching_08.jpg)

Prompt caching offers several advantages:

- Faster responses: Requests using cached content execute more quickly
- Lower costs: You pay less for the cached portions of your requests
- Automatic optimization: The initial request writes to the cache, follow-up requests read from it

However, there are important limitations to keep in mind:

- Cache duration: Cached content only lives for one hour
- Limited use cases: Only beneficial when you're repeatedly sending the same content
- High frequency requirement: Most effective when the same content appears extremely frequently in your requests

Prompt caching works best for scenarios like ***document analysis workflows***, where you're **asking multiple questions about the same large document, or iterative editing tasks where the base content remains constant while you refine specific aspects**.

# 6. Rules of prompt caching

## Cache Breakpoints

需要手動設置 Cache Breakpoint 來指定要 cache 的 Block

- You must manually add a 'cache breakpoint' to a block
- Work done for everything **before the breakpoint** will be cached
- Cache will only be used on follow-up requests if **the content up to and including the breakpoint is identical**

![](./figure/05/Rules_of_Prompt_Caching_01.jpg)

使用 Cache 需要將 request expand 為以下格式:
> use the expanded format with the `cache_control` field set to `{"type": "ephemeral"}`.

![](./figure/05/Rules_of_Prompt_Caching_02.jpg)

## How Cache Breakpoints Work

When you place a cache breakpoint in a message, Claude caches all the processing work up to and including that breakpoint. Content after the breakpoint is processed normally without caching.

![](./figure/05/Rules_of_Prompt_Caching_03.jpg)

For the cache to be useful in follow-up requests, the content **must be identical** up to the breakpoint. Even small changes like *adding the word "please"* will invalidate the cache and force Claude to reprocess everything.

![](./figure/05/Rules_of_Prompt_Caching_04.jpg)

## Cross-Message Caching

Cache breakpoints can **span across multiple messages and message types**. If you place a breakpoint in a later message, all previous messages (user, assistant, etc.) will be included in the cached content.

![](./figure/05/Rules_of_Prompt_Caching_05.jpg)

## System Prompts and Tools

不僅限於 Text Block，Breakpoint 也可用於:

- System prompts
- Tool definitions
- Image blocks
- Tool use and tool result blocks

> **System prompts** and **tool definitions** are excellent candidates for caching since they *rarely change between requests*. This is often where you'll get the most benefit from prompt caching.

![](./figure/05/Rules_of_Prompt_Caching_06.jpg)

## Cache Ordering

Claude processes your request components in a specific order: 
1. tools first
2. system prompt
3. messages

You can add **up to *four* cache breakpoints** total. This gives you flexibility in what gets cached when different parts of your request change.

![](./figure/05/Rules_of_Prompt_Caching_07.jpg)
![](./figure/05/Rules_of_Prompt_Caching_08.jpg)

## Minimum Content Length

Content must be **at least `1024` tokens** long to be cached. This is the **sum of all messages and blocks** you're trying to cache, not individual blocks.

![](./figure/05/Rules_of_Prompt_Caching_09.jpg)

> The key to effective prompt caching is identifying which parts of your requests stay consistent across multiple calls and placing breakpoints strategically to maximize reuse while minimizing cache invalidation.

# 7. Prompt caching in action

> 程式碼詳見 : [./source/003_caching.ipynb](./source/003_caching.ipynb)

## Setting Up Tool Schema Caching

Add a cache control field to the last tool in your list

```python
if tools:
    tools_clone = tools.copy()
    last_tool = tools_clone[-1].copy()
    last_tool["cache_control"] = {"type": "ephemeral"}
    tools_clone[-1] = last_tool
    params["tools"] = tools_clone
```

> While you could directly modify ```tools[-1]["cache_control"]```. 
> This the copying approach prevents issues if you later reorder your tools.

## System Prompt Caching

For system prompts, you need to structure them as a text block with cache control:

```python
if system:
    params["system"] = [
        {
            "type": "text",
            "text": system,
            "cache_control": {"type": "ephemeral"}
        }
    ]
```

This converts your system prompt from a simple string into a structured format that supports caching.

## Cache Ordering and Breakpoints
You can set multiple cache breakpoints in a single request. The order matters:

1. Tools (if provided)
2. System prompt (if provided)
3. Messages

If you change your system prompt but keep the same tools, you'll see a partial cache read (for tools) and a cache write (for the new system prompt). This granular caching means you only pay for processing the parts that actually changed.

剩下的部分到 [./source/003_caching.ipynb](./source/003_caching.ipynb)

# 8. Code execution and the Files API

## Files API

Instead of encoding images or PDFs directly in your messages as `base64` data, you can upload files ahead of time and reference them later.

![](./figure/05/Code_Execution_and_the_Files_API_01.jpg)

1. Upload your file (image, PDF, text, etc.) to Claude using a separate API call
2. Receive a file metadata object containing a unique file ID
3. Reference that file ID in future messages instead of including raw file data

![](./figure/05/Code_Execution_and_the_Files_API_02.jpg)

## Code Execution Tool

Code execution is a `server-based` tool that doesn't require you to provide an implementation. 
You simply include a **predefined tool schema** in your request, and Claude can **optionally execute Python code in an *isolated Docker container***.

Key characteristics of the code execution environment:
- Runs in an isolated Docker container
- **No network access (can't make external API calls)**
- Claude **can execute code multiple times during a single conversation**
- Results are captured and interpreted by Claude for the final response

![](./figure/05/Code_Execution_and_the_Files_API_03.jpg)

## Combining Files API and Code Execution

Here's a typical workflow:

1. Upload your data file (like a CSV) using the Files API
2. Include a container upload block in your message with the file ID
3. Ask Claude to analyze the data
4. Claude writes and executes code to process your file
5. Claude can generate outputs (like plots) that you can download

![](./figure/05/Code_Execution_and_the_Files_API_04.jpg)

## Practical Example

Let's look at a real example using streaming service data.

The CSV file contains user information including subscription tiers, viewing habits, and whether they've churned (canceled their subscription).

> 完整程式碼請看 [./source/005_code_execution.ipynb](./source/005_code_execution.ipynb)